# 뉴스 사실검증 데이터 전처리

`AI_기반_뉴스_사실검증_시스템_프로젝트_데이터_(1).csv`를 분석 가능한 형태로 정제합니다.

전처리 대상은 **`기사 본문 전체` 컬럼만**입니다. 제목·작성일·URL·레이블은 변경하지 않으며, 정제 결과는 `기사 본문 전처리`라는 새 컬럼에 저장합니다.

In [28]:
import pandas as pd
import numpy as np
import html
import re
import unicodedata

In [22]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. 데이터 불러오기

In [23]:
from pathlib import Path

csv_path = Path("/content/drive/MyDrive/멋사/data/AI_기반_뉴스_사실검증_시스템_프로젝트_데이터.csv")
df_raw = pd.read_csv(csv_path, encoding="utf-8")

In [24]:
df_raw.head()

,기사제목,작성일,URL,기사 본문 전체,검색 구분 레이블
0,"9명 가족 잃고, 하염없이 기다리던 푸딩이... 동물단체가 구조",2025-01-01,https://www.chosun.com/national/national_general/2025/01/01/7S62UVQCONDSPA5G4OQF3NYSDI,"9명 가족 잃고, 하염없이 기다리던 푸딩이... 동물단체가 구조 이혜진 기자 입력 2025.01.01. 23:47 업데이트 2025.01.02. 02:53 5 무안국제공항 제주항공 여객기 참사로 일가족 9명을...",False
1,"폴크스바겐, 전기차 정보 부실 관리 논란",2025-01-01,https://www.chosun.com/economy/auto/2025/01/01/S7LNVYUONNHU3KF3WJWFZZ3NVE,"폴크스바겐, 전기차 정보 부실 관리 논란 80만대 위치 정보와 개인 정보 암호화 안 한 채 온라인에 방치 조재희 기자 입력 2025.01.01. 00:35 업데이트 2025.01.02. 16:33 3 전기차 ...",True
2,최저임금 1만30원으로 인상… 육아휴직 급여 최대 월 250만원,2025-01-01,https://www.chosun.com/economy/economy_general/2025/01/01/ADO7O3Q4WJBCJAX6UUEY4MSBV4,최저임금 1만30원으로 인상… 육아휴직 급여 최대 월 250만원 새해부터 달라지는 정책·제도 김희래 기자 입력 2025.01.01. 01:28 업데이트 2025.01.02. 16:12 1 ◇노동·복지·가정 못...,True
3,‘직원 한 명에 로봇 수십 대’… 이젠 로봇이 공장 움직인다,2025-01-01,https://www.chosun.com/economy/economy_general/2025/01/01/TGA2SHZ4SFACHOHRP36OUPCMHE,2026년 4월 20일(월) 신문구독 | English | 日本語 | 中文 34 조선경제 오피니언 정치 사회 국제 건강 스포츠 문화·연예 콘텐츠판 땅집고 BEMIL 군사세계 헬스조선 IT조선 조선에듀 어린이조...,True
4,"정부, 美 ‘CES 2025′에 역대 최대 통합한국관 운영",2025-01-01,https://www.chosun.com/economy/industry-company/2025/01/01/2LIJLNBNSNANXCA6WLEDQ7A74A,2026년 4월 20일(월) 신문구독 | English | 日本語 | 中文 3 조선경제 오피니언 정치 사회 국제 건강 스포츠 문화·연예 콘텐츠판 땅집고 BEMIL 군사세계 헬스조선 IT조선 조선에듀 어린이조선...,True


## 2. 원본 데이터 품질 진단

In [25]:
def quality_report(frame: pd.DataFrame) -> pd.DataFrame:
    report = pd.DataFrame({
        "dtype": frame.dtypes.astype(str),
        "결측치": frame.isna().sum(),
        "결측률(%)": frame.isna().mean().mul(100).round(2),
        "고유값": frame.nunique(dropna=True),
    })
    return report

display(quality_report(df_raw))
print(f"완전 중복 행: {df_raw.duplicated().sum():,}건")
print(f"중복 URL(첫 행 제외): {df_raw['URL'].duplicated().sum():,}건")
print(f"중복 제목(첫 행 제외): {df_raw['기사제목'].duplicated().sum():,}건")
display(df_raw["검색 구분 레이블"].value_counts(dropna=False).rename("건수").to_frame())

# 첫 번째 행까지 포함한 URL 중복 그룹 전체
duplicate_url_rows = (
    df_raw.loc[df_raw["URL"].notna() & df_raw.duplicated(subset=["URL"], keep=False)]
    .sort_values(["URL", "작성일"], kind="stable")
    .reset_index(names="원본_index")
)
print(f"\n중복 URL 그룹에 속한 행: {len(duplicate_url_rows):,}건")
display(duplicate_url_rows)

# 첫 번째 행까지 포함한 제목 중복 그룹 전체
duplicate_title_rows = (
    df_raw.loc[df_raw["기사제목"].notna() & df_raw.duplicated(subset=["기사제목"], keep=False)]
    .sort_values(["기사제목", "작성일", "URL"], kind="stable")
    .reset_index(names="원본_index")
)
print(f"\n중복 제목 그룹에 속한 행: {len(duplicate_title_rows):,}건")
display(duplicate_title_rows)

,dtype,결측치,결측률(%),고유값
기사제목,object,0,0.00,2694
작성일,object,0,0.00,244
URL,object,0,0.00,2696
기사 본문 전체,object,1,0.04,2704
검색 구분 레이블,bool,0,0.00,2


완전 중복 행: 0건
중복 URL(첫 행 제외): 10건
중복 제목(첫 행 제외): 12건


,건수
검색 구분 레이블,
True,2507
False,199



중복 URL 그룹에 속한 행: 20건


,원본_index,기사제목,작성일,URL,기사 본문 전체,검색 구분 레이블
0,507,﻿“국민연금 못 받을라”... 수급자 41만명 늘때 가입자 57만명 줄어,2025-02-09,https://www.chosun.com/economy/economy_general/2025/02/09/OQKNQDJLTBDB3GZ43E2OZO4HII,2026년 4월 21일(화) 신문구독 | English | 日本語 | 中文 13 조선경제 오피니언 정치 사회 국제 건강 스포츠 문화·연예 콘텐츠판 땅집고 BEMIL 군사세계 헬스조선 IT조선 조선에듀 어린이조...,False
1,510,﻿“국민연금 못 받을라”... 수급자 41만명 늘때 가입자 57만명 줄어,2025-02-09,https://www.chosun.com/economy/economy_general/2025/02/09/OQKNQDJLTBDB3GZ43E2OZO4HII,2026년 4월 21일(화) 신문구독 | English | 日本語 | 中文 13 조선경제 오피니언 정치 사회 국제 건강 스포츠 문화·연예 콘텐츠판 땅집고 BEMIL 군사세계 헬스조선 IT조선 조선에듀 어린이조...,True
2,546,미국 1월 소비자물가 3% 상승... 금리 인하 속도 늦춰질 듯,2025-02-12,https://www.chosun.com/economy/economy_general/2025/02/12/6KAQJONZLRB4HIL6R2ER3BJ5WQ,미국 1월 소비자물가 3% 상승... 금리 인하 속도 늦춰질 듯 김정훈 기자 입력 2025.02.12. 22:36 업데이트 2025.02.13. 11:00 1 미국의 1월 소비자물가지수가 전년 동월 대비 3%...,False
3,551,미국 1월 소비자물가 3% 상승... 금리 인하 속도 늦춰질 듯,2025-02-12,https://www.chosun.com/economy/economy_general/2025/02/12/6KAQJONZLRB4HIL6R2ER3BJ5WQ,미국 1월 소비자물가 3% 상승... 금리 인하 속도 늦춰질 듯 김정훈 기자 입력 2025.02.12. 22:36 업데이트 2025.02.13. 11:00 1 미국의 1월 소비자물가지수가 전년 동월 대비 3%...,True
4,962,올해 세 번째 아프리카돼지열병(ASF)...경기 양주에서 발생,2025-03-16,https://www.chosun.com/economy/economy_general/2025/03/16/FVMTLUKQ2FDFPMY5WGSHXODXVA,올해 세 번째 아프리카돼지열병(ASF)...경기 양주에서 발생 1월에 이어 양주에서만 올 들어 3회 발생 인접 6개 시군 축산시설 종사자·차량 일시 이동 중지 조재희 기자 입력 2025.03.16. 23:56...,False
5,964,올해 세 번째 아프리카돼지열병(ASF)...경기 양주에서 발생,2025-03-16,https://www.chosun.com/economy/economy_general/2025/03/16/FVMTLUKQ2FDFPMY5WGSHXODXVA,올해 세 번째 아프리카돼지열병(ASF)...경기 양주에서 발생 1월에 이어 양주에서만 올 들어 3회 발생 인접 6개 시군 축산시설 종사자·차량 일시 이동 중지 조재희 기자 입력 2025.03.16. 23:56...,True
6,1443,"IMF, 올해 한국 성장률 전망 석 달만에 2%→1%로 낮춰",2025-04-22,https://www.chosun.com/economy/economy_general/2025/04/22/NSCGHPW7KBHIDEWPG6AXEHNZUU,"IMF, 올해 한국 성장률 전망 석 달만에 2%→1%로 낮춰 트럼프發 관세 전쟁 충격 반영 세계 경제도 3.3→2.8%로 내려 강우량 기자 입력 2025.04.22. 22:00 업데이트 2025.04.23. ...",False
7,1450,"IMF, 올해 한국 성장률 전망 석 달만에 2%→1%로 낮춰",2025-04-22,https://www.chosun.com/economy/economy_general/2025/04/22/NSCGHPW7KBHIDEWPG6AXEHNZUU,"IMF, 올해 한국 성장률 전망 석 달만에 2%→1%로 낮춰 트럼프發 관세 전쟁 충격 반영 세계 경제도 3.3→2.8%로 내려 강우량 기자 입력 2025.04.22. 22:00 업데이트 2025.04.23. ...",True
8,1456,IMF “관세 전쟁에 2년 후 세계 부채 비율 117%...2차 대전 수준의 빚더미”,2025-04-23,https://www.chosun.com/economy/economy_general/2025/04/23/FA5D4247IND5BDMIKI2LCAPKMQ,2026년 4월 21일(화) 신문구독 | English | 日本語 | 中文 0 조선경제 오피니언 정치 사회 국제 건강 스포츠 문화·연예 콘텐츠판 땅집고 BEMIL 군사세계 헬스조선 IT조선 조선에듀 어린이조선...,False
9,1465,IMF “관세 전쟁에 2년 후 세계 부채 비율 117%...2차 대전 수준의 빚더미”,2025-04-23,https://www.chosun.com/economy/economy_general/2025/04/23/FA5D4247IND5BDMIKI2LCAPKMQ,2026년 4월 21일(화) 신문구독 | English | 日本語 | 中文 0 조선경제 오피니언 정치 사회 국제 건강 스포츠 문화·연예 콘텐츠판 땅집고 BEMIL 군사세계 헬스조선 IT조선 조선에듀 어린이조선...,True



중복 제목 그룹에 속한 행: 23건


,원본_index,기사제목,작성일,URL,기사 본문 전체,검색 구분 레이블
0,2496,1180회 로또 1등 11명…인터넷 구매로 25억원 당첨,2025-07-12,https://www.chosun.com/economy/money/2025/07/12/RRY3IQT7FVG5VMZ77TBJKVJXMU,"1180회 로또 1등 11명…인터넷 구매로 25억원 당첨 김태호 기자(조선비즈) 입력 2025.07.12. 21:47 0 동행복권은 제1180회 로또복권 추첨에서 6, 12, 18, 37, 40, 41이 1등...",False
1,2504,1180회 로또 1등 11명…인터넷 구매로 25억원 당첨,2025-07-12,https://www.chosun.com/economy/money/2025/07/12/RRY3IQT7FVG5VMZ77TBJKVJXMU,"1180회 로또 1등 11명…인터넷 구매로 25억원 당첨 김태호 기자(조선비즈) 입력 2025.07.12. 21:47 0 동행복권은 제1180회 로또복권 추첨에서 6, 12, 18, 37, 40, 41이 1등...",True
2,1456,IMF “관세 전쟁에 2년 후 세계 부채 비율 117%...2차 대전 수준의 빚더미”,2025-04-23,https://www.chosun.com/economy/economy_general/2025/04/23/FA5D4247IND5BDMIKI2LCAPKMQ,2026년 4월 21일(화) 신문구독 | English | 日本語 | 中文 0 조선경제 오피니언 정치 사회 국제 건강 스포츠 문화·연예 콘텐츠판 땅집고 BEMIL 군사세계 헬스조선 IT조선 조선에듀 어린이조선...,False
3,1465,IMF “관세 전쟁에 2년 후 세계 부채 비율 117%...2차 대전 수준의 빚더미”,2025-04-23,https://www.chosun.com/economy/economy_general/2025/04/23/FA5D4247IND5BDMIKI2LCAPKMQ,2026년 4월 21일(화) 신문구독 | English | 日本語 | 中文 0 조선경제 오피니언 정치 사회 국제 건강 스포츠 문화·연예 콘텐츠판 땅집고 BEMIL 군사세계 헬스조선 IT조선 조선에듀 어린이조선...,True
4,1443,"IMF, 올해 한국 성장률 전망 석 달만에 2%→1%로 낮춰",2025-04-22,https://www.chosun.com/economy/economy_general/2025/04/22/NSCGHPW7KBHIDEWPG6AXEHNZUU,"IMF, 올해 한국 성장률 전망 석 달만에 2%→1%로 낮춰 트럼프發 관세 전쟁 충격 반영 세계 경제도 3.3→2.8%로 내려 강우량 기자 입력 2025.04.22. 22:00 업데이트 2025.04.23. ...",False
5,1450,"IMF, 올해 한국 성장률 전망 석 달만에 2%→1%로 낮춰",2025-04-22,https://www.chosun.com/economy/economy_general/2025/04/22/NSCGHPW7KBHIDEWPG6AXEHNZUU,"IMF, 올해 한국 성장률 전망 석 달만에 2%→1%로 낮춰 트럼프發 관세 전쟁 충격 반영 세계 경제도 3.3→2.8%로 내려 강우량 기자 입력 2025.04.22. 22:00 업데이트 2025.04.23. ...",True
6,193,“환율 1400원대 후반 된 건 정치 불안 때문”,2025-01-15,https://www.chosun.com/economy/economy_general/2025/01/15/RAR42MQ3JJEL5N47NMMPHS3OIU,2026년 4월 21일(화) 신문구독 | English | 日本語 | 中文 4 조선경제 오피니언 정치 사회 국제 건강 스포츠 문화·연예 콘텐츠판 땅집고 BEMIL 군사세계 헬스조선 IT조선 조선에듀 어린이조선...,True
7,194,“환율 1400원대 후반 된 건 정치 불안 때문”,2025-01-15,https://www.chosun.com/economy/economy_general/2025/01/15/SGLS4PCKPZH5ZNNEAZU4YYUGUI,2026년 4월 21일(화) 신문구독 | English | 日本語 | 中文 0 조선경제 오피니언 정치 사회 국제 건강 스포츠 문화·연예 콘텐츠판 땅집고 BEMIL 군사세계 헬스조선 IT조선 조선에듀 어린이조선...,True
8,1716,美 4월 소비자물가 상승률 2.3%... 4년여 만에 최저,2025-05-13,https://www.chosun.com/economy/economy_general/2025/05/13/J2AKL3ZEOVGBNBRO234LUR7CKA,美 4월 소비자물가 상승률 2.3%... 4년여 만에 최저 김정훈 기자 입력 2025.05.13. 22:32 업데이트 2025.05.13. 22:50 1 4월 미국 소비자물가가 예상보다 낮게 상승한 것으로 집...,False
9,1721,美 4월 소비자물가 상승률 2.3%... 4년여 만에 최저,2025-05-13,https://www.chosun.com/economy/economy_general/2025/05/13/J2AKL3ZEOVGBNBRO234LUR7CKA,美 4월 소비자물가 상승률 2.3%... 4년여 만에 최저 김정훈 기자 입력 2025.05.13. 22:32 업데이트 2025.05.13. 22:50 1 4월 미국 소비자물가가 예상보다 낮게 상승한 것으로 집...,True


## 3. 기사 본문 전처리

전처리 규칙은 다음 세 가지입니다. ① 본문 시작부터 기사 작성 날짜(및 바로 이어지는 업데이트 날짜)까지 제거, ② 기자 이름 제거, ③ `100자평`부터 이후 텍스트 제거. 다른 원본 컬럼과 행 수는 변경하지 않습니다.

In [40]:
BODY_COLUMN = "기사 본문 전체"
CLEAN_BODY_COLUMN = "기사 본문 전처리"
MIN_SECTION_POSITION = 200

RELATED_ARTICLE_MARKERS = [
    "더보기", "관련기사", "관련 기사", "추천기사", "추천 기사",
    "AI 추천", "오늘의 멤버십", "많이 본 뉴스", "당신이 좋아할 만한 콘텐츠",
]
ADVERTISEMENT_MARKERS = [
    "By Taboola", "ADVERTISEMENT", "광고 닫기", "광고 정보",
    "AD 돌아가기", "지금 뜨는 콘텐츠",
]
COMMENT_MARKERS = [
    "100자평", "댓글", "최신순", "관심순",
    "한마디", "도움말 삭제기준",
]

SCRIPT_STYLE_RE = re.compile(r"<(script|style|noscript)[^>]*>.*?</\1>", re.IGNORECASE | re.DOTALL)
HTML_COMMENT_RE = re.compile(r"<!--.*?-->", re.DOTALL)
HTML_TAG_RE = re.compile(r"<[^>]+>")
MULTISPACE_RE = re.compile(r"\s+")
REPORTER_RE = re.compile(r"[가-힣]{2,5}(?:\s*[·ㆍ,]\s*[가-힣]{2,5})*\s*기자")
DATE_TOKEN = r"\d{4}[./-]\d{1,2}[./-]\d{1,2}\.?(?:\s*(?:오전|오후)?\s*\d{1,2}:\d{2}(?::\d{2})?)?"
META_DATE_RE = re.compile(
    rf"(?:기사\s*)?(?:입력|작성|등록|업데이트|수정|최종수정)\s*[:：]?\s*{DATE_TOKEN}",
    re.IGNORECASE,
)
LEADING_HEADER_RE = re.compile(
    rf"^.*?(?:(?:기사\s*)?(?:입력|작성|등록)\s*[:：]?\s*{DATE_TOKEN})"
    rf"(?:\s*(?:업데이트|수정|최종수정)\s*[:：]?\s*{DATE_TOKEN})*",
    re.IGNORECASE | re.DOTALL,
)
DECORATIVE_SYMBOL_RE = re.compile(r"[◇◆◈■□▲△▼▽▶▷◀◁●○◎※★☆♠♣♥♦]+")
LEADING_UI_NUMBER_RE = re.compile(r"^\s*\d{1,4}\s+(?=[“\"'‘A-Za-z가-힣])")
BRACKET_UI_NUMBER_RE = re.compile(
    r"^(.{0,300}?[\]】〕〉》])\s+\d{1,4}\s+(?=[“\"'‘A-Za-z가-힣])", re.DOTALL
)
DUPLICATE_TRAILING_TAG_RE = re.compile(r"\s*#([가-힣A-Za-z0-9_-]+)\s+\1\s*$")
TRAILING_HASHTAG_RE = re.compile(r"(?:\s*#[가-힣A-Za-z0-9_-]+)+\s*$")

def normalize_whitespace(text: str) -> str:
    return MULTISPACE_RE.sub(" ", text).strip()

BOUNDARY_TRANSLATION = str.maketrans({
    "‘": "'", "’": "'", "‚": "'", "‛": "'",
    "“": '"', "”": '"', "„": '"', "‟": '"',
    "—": "-", "–": "-",
})

def normalize_boundary_text(text: str) -> str:
    # 제목 경계 비교를 위해 따옴표·대시·말줄임표 표기 차이를 통일합니다.
    text = text.translate(BOUNDARY_TRANSLATION)
    text = re.sub(r"\.{2,}", "…", text)
    return normalize_whitespace(text)

def remove_control_characters(text: str) -> str:
    return "".join(char for char in text if char in {"\n", "\t"} or unicodedata.category(char) not in {"Cc", "Cf"})

def cut_at_markers(text: str, markers: list[str], min_position: int = MIN_SECTION_POSITION) -> str:
    positions = [text.find(marker) for marker in markers]
    positions = [position for position in positions if position >= min_position]
    return text[:min(positions)] if positions else text

def find_title_start(text: str, title: object) -> int:
    if pd.isna(title):
        return -1
    normalized_title = normalize_boundary_text(unicodedata.normalize("NFKC", str(title)))
    return text.find(normalized_title) if normalized_title else -1

def cut_through_article_title(text: str, title: object) -> tuple[str, bool]:
    # CSV의 기사제목을 경계로 메뉴·날짜·언어 선택 등 앞쪽 UI를 한 번에 제거합니다.
    if pd.isna(title):
        return text, False
    normalized_title = normalize_boundary_text(unicodedata.normalize("NFKC", str(title)))
    position = text.find(normalized_title)
    if position < 0:
        return text, False
    return text[position + len(normalized_title):].lstrip(" -–—:|"), True

# 1. HTML 및 웹 UI 마크업 제거
def remove_html_ui(text: str) -> str:
    text = html.unescape(text)
    text = SCRIPT_STYLE_RE.sub(" ", text)
    text = HTML_COMMENT_RE.sub(" ", text)
    return HTML_TAG_RE.sub(" ", text)

# 2. 기자명과 입력·업데이트·수정 날짜 제거
def remove_reporter_information(text: str) -> str:
    header = LEADING_HEADER_RE.match(text)
    if header and header.end() <= 500:
        text = text[header.end():]
    text = REPORTER_RE.sub(" ", text)
    return META_DATE_RE.sub(" ", text)

def remove_leading_ui_number(text: str) -> str:
    # 기사 첫 문장 앞의 단독 숫자만 대상으로 하며 본문 전체 숫자는 건드리지 않습니다.
    text = LEADING_UI_NUMBER_RE.sub("", text, count=1)
    return BRACKET_UI_NUMBER_RE.sub(r"\1 ", text, count=1)

# 3. 관련기사·추천기사 영역 제거
def remove_related_articles(text: str) -> str:
    return cut_at_markers(text, RELATED_ARTICLE_MARKERS)

# 4. 광고 영역 제거
def remove_advertisements(text: str) -> str:
    return cut_at_markers(text, ADVERTISEMENT_MARKERS)

# 5. 댓글·100자평 영역 제거
def remove_comments(text: str) -> str:
    return cut_at_markers(text, COMMENT_MARKERS)

def remove_trailing_tags(text: str) -> str:
    # 예: '#로봇 로봇' 또는 '#로봇 #스마트팩토리'
    text = DUPLICATE_TRAILING_TAG_RE.sub("", text)
    return TRAILING_HASHTAG_RE.sub("", text)

def clean_article_body(body: object, title: object = pd.NA) -> object:
    if pd.isna(body):
        return pd.NA
    text = unicodedata.normalize("NFKC", str(body))
    text = remove_html_ui(text)
    text = remove_control_characters(text)
    text = DECORATIVE_SYMBOL_RE.sub(" ", text)
    text = normalize_boundary_text(text)
    text, _ = cut_through_article_title(text, title)
    text = remove_reporter_information(text)
    text = normalize_whitespace(text)
    text = remove_leading_ui_number(text)
    text = remove_related_articles(text)
    text = remove_advertisements(text)
    text = remove_comments(text)
    text = remove_trailing_tags(text)
    text = normalize_whitespace(text)
    return text if text else pd.NA

In [41]:
df = df_raw.copy()
title_boundary_found = pd.Series([
    False if pd.isna(body) else find_title_start(
        normalize_boundary_text(remove_control_characters(remove_html_ui(unicodedata.normalize("NFKC", str(body))))), title
    ) >= 0
    for body, title in zip(df[BODY_COLUMN], df["기사제목"])
])
df[CLEAN_BODY_COLUMN] = [
    clean_article_body(body, title)
    for body, title in zip(df[BODY_COLUMN], df["기사제목"])
]
df[CLEAN_BODY_COLUMN] = df[CLEAN_BODY_COLUMN].astype("string")

# 전처리 강도를 확인하는 진단 컬럼
df["본문 원문 길이"] = df[BODY_COLUMN].astype("string").str.len().astype("Int64")
df["본문 전처리 길이"] = df[CLEAN_BODY_COLUMN].str.len().astype("Int64")
df["본문 제거 비율"] = (
    1 - df["본문 전처리 길이"].div(df["본문 원문 길이"].replace(0, pd.NA))
).clip(lower=0, upper=1).round(4)

print(f"전체 행: {len(df):,}건")
print(f"기사제목 기준 시작점 탐지: {title_boundary_found.sum():,}건 ({title_boundary_found.mean():.1%})")
print(f"원문 본문 결측: {df[BODY_COLUMN].isna().sum():,}건")
print(f"전처리 본문 결측: {df[CLEAN_BODY_COLUMN].isna().sum():,}건")
display(df[["기사제목", BODY_COLUMN, CLEAN_BODY_COLUMN]].head(3))

전체 행: 2,706건
기사제목 기준 시작점 탐지: 2,643건 (97.7%)
원문 본문 결측: 1건
전처리 본문 결측: 3건


,기사제목,기사 본문 전체,기사 본문 전처리
0,"9명 가족 잃고, 하염없이 기다리던 푸딩이... 동물단체가 구조","9명 가족 잃고, 하염없이 기다리던 푸딩이... 동물단체가 구조 이혜진 기자 입력 2025.01.01. 23:47 업데이트 2025.01.02. 02:53 5 무안국제공항 제주항공 여객기 참사로 일가족 9명을...",무안국제공항 제주항공 여객기 참사로 일가족 9명을 잃은 반려견 '푸딩이'가 구조됐다. 동물권보호단체 케어는 지난달 31일 공식 인스타그램을 통해 보호자 없이 마을을 배회하던 푸딩이를 안전하게 보호 중이라고 밝...
1,"폴크스바겐, 전기차 정보 부실 관리 논란","폴크스바겐, 전기차 정보 부실 관리 논란 80만대 위치 정보와 개인 정보 암호화 안 한 채 온라인에 방치 조재희 기자 입력 2025.01.01. 00:35 업데이트 2025.01.02. 16:33 3 전기차 ...",전기차 전환에 뒤처지며 창사 87년 만에 처음으로 독일 자국 내 공장 폐쇄에 나선 폴크스바겐이 이번엔 전기차 약 80만대의 운행 데이터와 소유주 정보를 온라인에서 부실 관리했다는 논란에 휩싸였다. 전기차에 이...
2,최저임금 1만30원으로 인상… 육아휴직 급여 최대 월 250만원,최저임금 1만30원으로 인상… 육아휴직 급여 최대 월 250만원 새해부터 달라지는 정책·제도 김희래 기자 입력 2025.01.01. 01:28 업데이트 2025.01.02. 16:12 1 ◇노동·복지·가정 못...,"노동·복지·가정 못 받은 양육비, 정부가 선지급… 국가 검진에 C형 간염도 포함 최저임금 시간당 1만30원 =최저임금이 시간당 9860원에서 1만30원으로 1.7% 인상된다. 주 근로시간 40시간을 기준으로 ..."


In [42]:
df.to_csv("/content/drive/MyDrive/멋사/data/AI_기반_뉴스_사실검증_시스템_프로젝트_데이터_전처리.csv", index=False, encoding="utf-8-sig")

## 4. 전처리 결과 검증

In [43]:
# 본문 외 원본 컬럼이 바뀌지 않았는지 확인합니다.
for column in df_raw.columns:
    pd.testing.assert_series_equal(df[column], df_raw[column], check_names=True)
assert len(df) == len(df_raw), "전처리 중 행 수가 변경되었습니다."
assert df[CLEAN_BODY_COLUMN].dropna().str.strip().ne("").all(), "빈 문자열이 남아 있습니다."
clean_text = df[CLEAN_BODY_COLUMN].fillna("")
assert not clean_text.str.contains(REPORTER_RE).any(), "기자명이 남아 있습니다."
assert not clean_text.str.contains(META_DATE_RE).any(), "입력/업데이트 관련 날짜가 남아 있습니다."
assert not clean_text.str.contains(DECORATIVE_SYMBOL_RE).any(), "장식 기호가 남아 있습니다."
assert not clean_text.map(lambda text: bool(DUPLICATE_TRAILING_TAG_RE.search(text))).any(), "중복 말미 태그가 남아 있습니다."
assert not clean_text.str.contains(TRAILING_HASHTAG_RE).any(), "말미 해시태그가 남아 있습니다."
all_tail_markers = RELATED_ARTICLE_MARKERS + ADVERTISEMENT_MARKERS + COMMENT_MARKERS
has_residual_tail = clean_text.map(lambda text: any(text.find(marker) >= MIN_SECTION_POSITION for marker in all_tail_markers))
assert not has_residual_tail.any(), "관련기사·광고·댓글 영역 표식이 남아 있습니다."
print("검증 통과: 행 수와 본문 외 모든 컬럼이 원본과 같습니다.")

display(df[["본문 원문 길이", "본문 전처리 길이", "본문 제거 비율"]].describe())
comparison_columns = ["기사제목", BODY_COLUMN, CLEAN_BODY_COLUMN, "본문 제거 비율"]
display(df.nlargest(5, "본문 제거 비율")[comparison_columns])

검증 통과: 행 수와 본문 외 모든 컬럼이 원본과 같습니다.


,본문 원문 길이,본문 전처리 길이,본문 제거 비율
count,2705.0,2703.0,2703.0
mean,15299.696118,1351.783944,0.776717
std,8051.546076,772.362551,0.294248
min,124.0,124.0,0.0
25%,14079.0,804.0,0.86305
50%,18162.0,1140.0,0.9164
75%,20750.0,1852.5,0.9467
max,33666.0,7145.0,0.9861


,기사제목,기사 본문 전체,기사 본문 전처리,본문 제거 비율
1548,"현대차, 美박람회서 대형 수소전기트럭 첫 공개","현대차, 美박람회서 대형 수소전기트럭 첫 공개 조선일보 입력 2025.04.30. 00:34 업데이트 2025.04.30. 10:32 0 현대차가 28일(현지 시각) 미국 캘리포니아주 애너하임에서 열린 청정 ...",현대차가 28일(현지 시각) 미국 캘리포니아주 애너하임에서 열린 청정 운송 수단 박람회 'ACT 엑스포 2025'에서 대형 수소전기트럭 '더 뉴 엑시언트'를 처음 공개했다. 2020년 출시한 '엑시언트'의 첫...,0.9861
2094,"기아, 군용 ‘중형표준차’ 양산 본격화",2026년 4월 21일(화) 신문구독 | English | 日本語 | 中文 1 조선경제 오피니언 정치 사회 국제 건강 스포츠 문화·연예 콘텐츠판 땅집고 BEMIL 군사세계 헬스조선 IT조선 조선에듀 어린이조선...,"기아는 10일 오토랜드 광주 하남공장에서 차세대 중형표준차(KMTV) 출고 기념식을 갖고, 본격적인 양산을 시작했다. 1977년 이후 '국군의 발' 역할을 해온 일명 '두돈반'을 48년 만에 대체하는 차세대 ...",0.9861
216,"에쓰오일, 희망 나눔 성금 20억원","에쓰오일, 희망 나눔 성금 20억원 조선일보 입력 2025.01.17. 00:51 0 에쓰오일은 16일 서울 중구 사회복지공동모금회에 ‘희망 2025 나눔 캠페인’ 성금 20억원을 전달했다고 밝혔다. 에쓰오일...",에쓰오일은 16일 서울 중구 사회복지공동모금회에 '희망 2025 나눔 캠페인' 성금 20억원을 전달했다고 밝혔다. 에쓰오일이 2004년부터 22년 동안 기부한 성금은 총 270억원에 이른다. close Adv...,0.986
470,"[기업 브리핑] 현대차 등 8사, 美 전기차 충전 서비스 출시","[기업 브리핑] 현대차 등 8사, 美 전기차 충전 서비스 출시 조선일보 입력 2025.02.06. 00:33 업데이트 2025.02.06. 10:14 0 현대차, 도요타, BMW 등 8개 완성차 업체가 설립한...","현대차, 도요타, BMW 등 8개 완성차 업체가 설립한 조인트벤처 '아이오나'가 4일 미국 노스캐롤라이나주 본사에서 전기차 고속 충전 서비스 출시 행사를 열었다. 아이오나는 북미에서 올해 충전기 1000기, ...",0.9858
1427,"신한은행, 부산에 디지털금융교육센터 ‘학이재’ 개관",2026년 4월 21일(화) 신문구독 | English | 日本語 | 中文 0 조선경제 오피니언 정치 사회 국제 건강 스포츠 문화·연예 콘텐츠판 땅집고 BEMIL 군사세계 헬스조선 IT조선 조선에듀 어린이조선...,"신한은행은 부산 부산진구에 디지털금융 교육센터 '신한 학이재 부산'을 개관했다고 20일 밝혔다. 지난 18일 열린 개관식에는 정상혁 신한은행장, 박형준 부산시장, 김미영 금융감독원 금융소비자보호처장, 문우택 ...",0.9854


## 5. 저장

In [ ]:
output_path = csv_path.with_name(f"{csv_path.stem}_본문전처리.csv")
df.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"저장 완료: {output_path}")

## 6. NCP CLOVA Studio로 기사 주장 추출

전처리된 `기사 본문 전처리`에서 검증 가능한 주장과 수치 정보를 추출합니다. `.env`의 `NCP_CLOVASTUDIO_API_KEY=` 뒤에 키를 입력한 후 실행하세요. 기본값은 비용 보호를 위해 1건 테스트 및 최대 10건 배치입니다.

In [47]:
import json
import os
import time
import uuid
from pathlib import Path

import requests

In [49]:
def load_env_file(path: Path) -> None:
    if not path.is_file():
        return
    for raw_line in path.read_text(encoding="utf-8-sig").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip("\"'"))

# 로컬 프로젝트와 Google Drive 데이터 폴더 양쪽을 지원합니다.
for env_path in [Path.cwd() / ".env", csv_path.parent / ".env"]:
    load_env_file(env_path)

NCP_API_KEY = os.getenv("NCP_CLOVASTUDIO_API_KEY", "").strip()
NCP_MODEL = "HCX-007"
NCP_API_URL = f"https://clovastudio.stream.ntruss.com/v3/chat-completions/{NCP_MODEL}"
MAX_CHARS_PER_CHUNK = 40_000
CHUNK_OVERLAP = 500
REQUEST_INTERVAL_SECONDS = 0.5

print("NCP API 키를 불러왔습니다." if NCP_API_KEY else "NCP API 키가 없습니다. .env에 입력한 후 다시 실행하세요.")

NCP API 키를 불러왔습니다.


In [52]:
CLAIM_SCHEMA = {
    "type": "object",
    "properties": {
        "claims": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "claim_text": {"type": "string", "description": "기사에 명시된 완결된 주장"},
                    "claim_type": {"type": "string", "enum": ["numeric", "non_numeric"]},
                    "subject": {"type": "string"},
                    "predicate": {"type": "string"},
                    "object": {"type": "string"},
                    "numeric_values": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "raw_value": {"type": "string"},
                                "normalized_value": {"type": "string"},
                                "unit": {"type": "string"},
                                "context": {"type": "string"}
                            },
                            "required": ["raw_value", "normalized_value", "unit", "context"]
                        }
                    },
                    "evidence_quote": {"type": "string", "description": "본문의 짧은 직접 근거 문구"}
                },
                "required": ["claim_text", "claim_type", "subject", "predicate", "object", "numeric_values", "evidence_quote"]
            }
        }
    },
    "required": ["claims"]
}

SYSTEM_PROMPT = """
당신은 뉴스 기사에서 통계 자료로 검증 가능한 Claim을 추출하는 분석가입니다.

목표:
뉴스 문장에서 통계표와 비교할 수 있는 독립적인 주장을 추출하고
지정된 구조로 변환합니다.

추출 대상:
- 특정 시점의 수치
- 비율 및 구성비
- 증가율과 감소율
- 두 대상 또는 두 시점의 비교
- 순위
- 최고·최저
- 일정 기간의 추세
- 이상·이하·초과·미만 등의 범위 주장

제외 대상:
- 주관적 평가
- 전망과 희망
- 수사적 표현
- 통계로 검증할 수 없는 기업 관계자의 개인 경험
- 단순 날짜, 주소, 인원 소개
- 의미가 불분명한 숫자

규칙:
1. 한 문장에 여러 Claim이 있으면 각각 분리합니다.
2. 원문에 없는 숫자나 단위를 생성하지 않습니다.
3. 이전 문장에서 보완한 정보는 inferred_fields에 기록합니다.
4. 원문에 직접 있는 정보는 explicit_fields에 기록합니다.
5. '약', '가량', '이상', '넘는' 등의 한정 표현을 보존합니다.
6. normalized_claim은 앞 문맥 없이도 이해 가능한 문장으로 작성합니다.
7. 검증이 불가능하면 verifiable_with_statistics를 false로 설정합니다.
8. 값이 없으면 null을 사용하고 빈 숫자를 임의로 채우지 않습니다.
"""

def split_article(text: str, max_chars: int = MAX_CHARS_PER_CHUNK, overlap: int = CHUNK_OVERLAP) -> list[str]:
    if len(text) <= max_chars:
        return [text]
    chunks, start = [], 0
    while start < len(text):
        end = min(start + max_chars, len(text))
        if end < len(text):
            boundary = max(text.rfind(". ", start, end), text.rfind("다. ", start, end))
            if boundary > start + max_chars // 2:
                end = boundary + 1
        chunks.append(text[start:end].strip())
        if end >= len(text):
            break
        start = max(end - overlap, start + 1)
    return chunks

def call_clova_claim_api(article_text: str, max_retries: int = 3) -> tuple[dict, dict]:
    if not NCP_API_KEY:
        raise RuntimeError("NCP_CLOVASTUDIO_API_KEY가 설정되지 않았습니다.")
    headers = {
        "Authorization": f"Bearer {NCP_API_KEY}",
        "X-NCP-CLOVASTUDIO-REQUEST-ID": str(uuid.uuid4()),
        "Content-Type": "application/json",
        "Accept": "application/json",
    }
    body = {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"다음 기사에서 주장을 추출하세요.\n\n{article_text}"},
        ],
        "topP": 0.8, "topK": 0, "temperature": 0.1,
        "repetitionPenalty": 1.1, "maxCompletionTokens": 4096,
        "thinking": {"effort": "none"},
        "responseFormat": {"type": "json", "schema": CLAIM_SCHEMA},
    }
    for attempt in range(max_retries):
        response = requests.post(NCP_API_URL, headers=headers, json=body, timeout=180)
        if response.status_code == 429 or response.status_code >= 500:
            if attempt + 1 < max_retries:
                time.sleep(2 ** attempt)
                continue
        response.raise_for_status()
        payload = response.json()
        result = payload.get("result", payload)
        content = result.get("message", {}).get("content", "")
        if not content:
            raise RuntimeError(f"CLOVA 응답에 content가 없습니다: {payload}")
        return json.loads(content), result.get("usage", {})
    raise RuntimeError("CLOVA API 재시도 횟수를 초과했습니다.")

def extract_claims_from_article(article_text: str) -> tuple[list[dict], dict]:
    unique, total_usage = {}, {"promptTokens": 0, "completionTokens": 0, "totalTokens": 0}
    for chunk_number, chunk in enumerate(split_article(article_text), 1):
        parsed, usage = call_clova_claim_api(chunk)
        for claim in parsed.get("claims", []):
            claim["chunk_number"] = chunk_number
            unique.setdefault(claim.get("claim_text", "").strip(), claim)
        for key in total_usage:
            total_usage[key] += int(usage.get(key, 0) or 0)
        time.sleep(REQUEST_INTERVAL_SECONDS)
    unique.pop("", None)
    return list(unique.values()), total_usage

### 6-1. 한 기사로 테스트

In [54]:
test_rows = df[df[CLEAN_BODY_COLUMN].notna()]
if not NCP_API_KEY:
    print("API 키를 입력한 뒤 설정 셀부터 다시 실행하세요.")
elif test_rows.empty:
    print("전처리된 본문이 없습니다.")
else:
    test_row = test_rows.iloc[1]
    test_claims, test_usage = extract_claims_from_article(test_row[CLEAN_BODY_COLUMN])
    print(f"추출 주장: {len(test_claims)}개 / 토큰 사용량: {test_usage}")
    display(pd.json_normalize(test_claims))

추출 주장: 1개 / 토큰 사용량: {'promptTokens': 809, 'completionTokens': 165, 'totalTokens': 974}


,claim_text,claim_type,subject,predicate,object,numeric_values,evidence_quote,chunk_number
0,폴크스바겐 그룹 전기차 80만대에서 수집한 데이터,numeric,폴크스바겐 그룹 전기차,수집한 데이터,80만대,"[{'raw_value': '80만대', 'normalized_value': '800,000', 'unit': '', 'context': '폴크스바겐 그룹 전기차'}]","폴크스바겐과 아우디, 세아트, 스코다 등 폴크스바겐그룹 전기차 80만대에서 수집한 수 테라바이트 규모 데이터가 지난 몇 달간 암호화되지 않은 채로 아마존 클라우드에 방치됐던 것으로 나타났다.",1


### 6-2. 배치 추출 및 체크포인트 저장

`MAX_ARTICLES`를 `None`으로 바꾸면 전체 기사를 처리합니다. 먼저 소량으로 비용과 결과를 확인하세요.

In [ ]:
MAX_ARTICLES = 10  # 전체 실행은 None
CLAIMS_OUTPUT_PATH = csv_path.with_name(f"{csv_path.stem}_주장추출.csv")
CHECKPOINT_PATH = csv_path.with_name(f"{csv_path.stem}_주장추출_checkpoint.jsonl")

def load_completed_urls(path: Path) -> set[str]:
    if not path.is_file():
        return set()
    completed = set()
    for line in path.read_text(encoding="utf-8").splitlines():
        try:
            record = json.loads(line)
            if record.get("status") == "success":
                completed.add(record.get("url", ""))
        except json.JSONDecodeError:
            continue
    return completed

if not NCP_API_KEY:
    print("API 키를 입력한 뒤 이 셀을 실행하세요.")
else:
    completed_urls = load_completed_urls(CHECKPOINT_PATH)
    targets = df[df[CLEAN_BODY_COLUMN].notna() & ~df["URL"].isin(completed_urls)]
    if MAX_ARTICLES is not None:
        targets = targets.head(MAX_ARTICLES)
    print(f"이번 실행 대상: {len(targets):,}건 / 기존 완료: {len(completed_urls):,}건")

    with CHECKPOINT_PATH.open("a", encoding="utf-8") as checkpoint:
        for sequence, (_, row) in enumerate(targets.iterrows(), 1):
            record = {"url": row["URL"], "title": row["기사제목"]}
            try:
                claims, usage = extract_claims_from_article(row[CLEAN_BODY_COLUMN])
                record.update({"status": "success", "claims": claims, "usage": usage})
                print(f"[{sequence}/{len(targets)}] 성공: {len(claims)}개 - {row['기사제목'][:40]}")
            except Exception as error:
                record.update({"status": "failed", "error": str(error), "claims": []})
                print(f"[{sequence}/{len(targets)}] 실패: {error}")
            checkpoint.write(json.dumps(record, ensure_ascii=False) + "\n")
            checkpoint.flush()

    flat_rows = []
    for line in CHECKPOINT_PATH.read_text(encoding="utf-8").splitlines():
        record = json.loads(line)
        if record.get("status") != "success":
            continue
        for claim_number, claim in enumerate(record.get("claims", []), 1):
            flat_rows.append({
                "URL": record["url"], "기사제목": record["title"],
                "claim_number": claim_number, **claim,
                "numeric_values_json": json.dumps(claim.get("numeric_values", []), ensure_ascii=False),
            })
    claims_df = pd.DataFrame(flat_rows)
    if "numeric_values" in claims_df.columns:
        claims_df = claims_df.drop(columns=["numeric_values"])
    claims_df.to_csv(CLAIMS_OUTPUT_PATH, index=False, encoding="utf-8-sig")
    print(f"저장 완료: {CLAIMS_OUTPUT_PATH} ({len(claims_df):,}개 주장)")

## 7. 기사 주장 추출